# Load and Clean Downloaded FX Price Data

Scratch notebook for pulling raw daily candles from `fx_candles.db` and eyeballing
recent data before building indicators on top of it.

- Fetches every stored daily candle for `INSTRUMENT` over the 10-year window
  `[START_YEAR, END_YEAR)`, i.e. up to but not including `END_YEAR`.
- Prints full detail for the most recent `LAST_N_DAYS` trading days.
- Re-run all cells to refresh against the latest data in `fx_candles.db`.

In [1]:
from __future__ import annotations

import sqlite3
from typing import Final

import pandas as pd

In [2]:
DB_PATH: Final[str] = "../../fx_candles.db"
TABLE: Final[str] = "candles_D"
INSTRUMENT: Final[str] = "EUR_USD"
START_YEAR: Final[int] = 2016
END_YEAR: Final[int] = 2026
LAST_N_DAYS: Final[int] = 90

In [ ]:
from pandas.core.frame import DataFrame
query = (
    f"SELECT * FROM {TABLE} "
    "WHERE instrument = ? AND time >= ? AND time < ? AND complete = 1 "
    "ORDER BY time ASC"
)
with sqlite3.connect(database=DB_PATH) as conn:
    candles: DataFrame = pd.read_sql_query(
        sql=query, con=conn, params=(INSTRUMENT, f"{START_YEAR}-01-01", f"{END_YEAR}-01-01")
    )

# print(candles.head())
print(f"Fetched {len(candles)} {INSTRUMENT} candles from {START_YEAR}-01-01 to {END_YEAR}-01-01 (exclusive).")

Fetched 2595 EUR_USD candles from 2016-01-01 to 2026-01-01 (exclusive).


In [ ]:
candles["date"] = pd.to_datetime(arg=candles["time"]).dt.date

# Get all unique dates in the candles DF -> sort them chronologically -> take the last N dates of them
last_90_dates = sorted(candles["date"].unique())[-LAST_N_DAYS:]
last_90_days = candles[candles["date"].isin(values=last_90_dates)]
# print(last_90_days)

print(f"Last {len(last_90_dates)} trading days: {last_90_dates[0]} → {last_90_dates[-1]}")

Last 90 trading days: 2025-08-26 → 2025-12-30


In [ ]:
with pd.option_context(pat="display.max_rows", pat=None, pat="display.max_columns", pat=None,
                        "display.width", None):
    print(last_90_days.to_string(index=False))

instrument                      time   bid_o   bid_h   bid_l   bid_c   ask_o   ask_h   ask_l   ask_c  volume  complete       date
   EUR_USD 2025-08-26T21:00:00+00:00 1.16436 1.16471 1.15732 1.16368 1.16478 1.16500 1.15749 1.16385  111719         1 2025-08-26
   EUR_USD 2025-08-27T21:00:00+00:00 1.16348 1.16968 1.16282 1.16816 1.16405 1.16982 1.16298 1.16834  112674         1 2025-08-27
   EUR_USD 2025-08-28T21:00:00+00:00 1.16793 1.17082 1.16499 1.16840 1.16856 1.17097 1.16517 1.16877  117171         1 2025-08-28
   EUR_USD 2025-08-31T21:00:00+00:00 1.16944 1.17357 1.16854 1.17106 1.16977 1.17374 1.16876 1.17121   82695         1 2025-08-31
   EUR_USD 2025-09-01T21:00:00+00:00 1.17082 1.17176 1.16121 1.16412 1.17105 1.17191 1.16137 1.16434  167778         1 2025-09-01
   EUR_USD 2025-09-02T21:00:00+00:00 1.16372 1.16814 1.16074 1.16603 1.16472 1.16829 1.16090 1.16621  127372         1 2025-09-02
   EUR_USD 2025-09-03T21:00:00+00:00 1.16577 1.16687 1.16292 1.16497 1.16659 1.16704 1.163

In [6]:
last_90_days.info()

<class 'pandas.core.frame.DataFrame'>
Index: 90 entries, 2505 to 2594
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instrument  90 non-null     object 
 1   time        90 non-null     object 
 2   bid_o       90 non-null     float64
 3   bid_h       90 non-null     float64
 4   bid_l       90 non-null     float64
 5   bid_c       90 non-null     float64
 6   ask_o       90 non-null     float64
 7   ask_h       90 non-null     float64
 8   ask_l       90 non-null     float64
 9   ask_c       90 non-null     float64
 10  volume      90 non-null     int64  
 11  complete    90 non-null     int64  
 12  date        90 non-null     object 
dtypes: float64(8), int64(2), object(3)
memory usage: 9.8+ KB


In [7]:
last_90_days.shape # output reads: number of rows, number of columns

(90, 13)

In [8]:
last_90_days.head()

,instrument,time,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c,volume,complete,date
2505,EUR_USD,2025-08-26T21:00:00+00:00,1.16436,1.16471,1.15732,1.16368,1.16478,1.16500,1.15749,1.16385,111719,1,2025-08-26
2506,EUR_USD,2025-08-27T21:00:00+00:00,1.16348,1.16968,1.16282,1.16816,1.16405,1.16982,1.16298,1.16834,112674,1,2025-08-27
2507,EUR_USD,2025-08-28T21:00:00+00:00,1.16793,1.17082,1.16499,1.16840,1.16856,1.17097,1.16517,1.16877,117171,1,2025-08-28
2508,EUR_USD,2025-08-31T21:00:00+00:00,1.16944,1.17357,1.16854,1.17106,1.16977,1.17374,1.16876,1.17121,82695,1,2025-08-31
2509,EUR_USD,2025-09-01T21:00:00+00:00,1.17082,1.17176,1.16121,1.16412,1.17105,1.17191,1.16137,1.16434,167778,1,2025-09-01


In [9]:
last_90_days.tail()

,instrument,time,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c,volume,complete,date
2590,EUR_USD,2025-12-23T22:00:00+00:00,1.17896,1.18072,1.17718,1.17748,1.17958,1.18088,1.17735,1.17792,89168,1,2025-12-23
2591,EUR_USD,2025-12-25T22:00:00+00:00,1.17755,1.17959,1.17608,1.17694,1.17855,1.17976,1.17624,1.17725,120235,1,2025-12-25
2592,EUR_USD,2025-12-28T22:00:00+00:00,1.17630,1.17884,1.17486,1.17717,1.17730,1.17900,1.17503,1.17735,110313,1,2025-12-28
2593,EUR_USD,2025-12-29T22:00:00+00:00,1.17668,1.17791,1.17426,1.17471,1.17751,1.17807,1.17442,1.17489,88246,1,2025-12-29
2594,EUR_USD,2025-12-30T22:00:00+00:00,1.17434,1.17587,1.17192,1.17449,1.17498,1.17604,1.17207,1.17469,98214,1,2025-12-30


In [11]:
week_day_1 = pd.Timestamp("2026-05-31").day_name()  # output reads: Sunday
week_day_2 = pd.Timestamp("2026-07-05").day_name()  # output reads: Sunday
print(week_day_1)
print(week_day_2)

Sunday
Sunday


In [12]:
pd.Timestamp("2025-12-25").day_name() # output reads: Thursday

'Thursday'